### Undisclosed Rebranding
This notebook focuses on undisclosed rebranding of certificates where companies use in their cerificates OpenSSL functions, but do not mention OpenSSL

In [1]:
from sec_certs.dataset.cc import CCDataset
from sec_certs.dataset.fips import FIPSDataset



In [15]:
print("Downloading CC dataset")

cc_dataset = CCDataset.from_web(auxiliary_datasets=True, artifacts=True, path="./datasets")

print(f"Downloaded {len(cc_dataset)} CC certificates")
print("Saving dataset")

#cc_dataset.to_json("./datasets/CC_dataset.json")

print("Dataset saved to ./datasets/CC_dataset.json")

print("Dowloading FIPS dataset")

fips_dataset = FIPSDataset.from_web()

print(f"Downloaded {len(fips_dataset)} FIPS 140 certificates")
print("Saving dataset")

fips_dataset.to_json("./datasets/FIPS_dataset.json")

print("Dataset saved to ./datasets/FIPS_dataset.json")

Downloading: CCDataset:   0%|          | 0.00/11.0G [00:00<?, ?B/s]

Dataset was created with sec-certs version 0.4.1.post1.dev71+g6b178753a (older than your version 0.4.1.post1.dev75+g9e07e5b06). To install the matching version: pip install sec-certs==0.4.1.post1.dev71+g6b178753a


Downloaded 6783 CC certificates
Saving dataset
Dataset saved to ./datasets/CC_dataset.json
Dowloading FIPS dataset


Downloading: FIPSDataset:   0%|          | 0.00/74.6M [00:00<?, ?B/s]

Downloaded 5477 FIPS 140 certificates
Saving dataset
Dataset saved to ./datasets/FIPS_dataset.json


In [ ]:
print("Loading datasets from JSON")
cc_dataset = CCDataset.from_json("./datasets/CC_dataset.json")
print(f"Loaded {len(cc_dataset)} CC certificates from JSON")
fips_dataset = FIPSDataset.from_json("./datasets/FIPS_dataset.json")
print(f"Loaded {len(fips_dataset)} FIPS 140 certificates from JSON")

### 1. Load and Filter OpenSSL Functions
We load the list of OpenSSL functions extracted from the docs and filter for specific API prefixes to avoid false positives (e.g., generic words like `connect`).

In [3]:
import json
import re

with open('src/openssl_man3.json', 'r') as f:
    all_functions = json.load(f)

prefixes = ('SSL_', 'EVP_', 'BIO_', 'BN_', 'X509_', 'PEM_', 'CMS_', 'CRYPTO_', 'ASN1_', 'd2i_', 'i2d_')
filtered_functions = [f for f in all_functions if f.startswith(prefixes)]
print(f"Total functions extracted: {len(all_functions)}")
print(f"Filtered high-confidence functions: {len(filtered_functions)}")


Total functions extracted: 5671
Filtered high-confidence functions: 3450


### 2. Define Extraction Logic
We define a function that checks if a certificate mentions OpenSSL via sec-certs keywords. If it doesn't, we scan its associated text files for our filtered OpenSSL functions.

In [10]:
import os
import pandas as pd
from pathlib import Path

def mentions_openssl_cc(cert):
    # Check if OpenSSL is in the CC report or ST keywords
    if cert.pdf_data.report_keywords and 'crypto_library' in cert.pdf_data.report_keywords:
        if 'OpenSSL' in cert.pdf_data.report_keywords['crypto_library']:
            return True
    if cert.pdf_data.st_keywords and 'crypto_library' in cert.pdf_data.st_keywords:
        if 'OpenSSL' in cert.pdf_data.st_keywords['crypto_library']:
            return True
    return False

def mentions_openssl_fips(cert):
    # Check if OpenSSL is in the FIPS policy keywords
    if cert.pdf_data.keywords and 'crypto_library' in cert.pdf_data.keywords:
        if 'OpenSSL' in cert.pdf_data.keywords['crypto_library']:
            return True
    return False

def scan_text_for_functions(text_path, functions):
    if not text_path or not os.path.exists(text_path):
        return {}
    
    with open(text_path, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()
        
    matches = {}
    for func in functions:
        # Use simple string matching for performance
        count = content.count(func)
        if count > 0:
            matches[func] = count
            
    return matches


### 3. Scan Common Criteria Dataset

In [17]:
results = []

print("Scanning CC Dataset...")
for cert in cc_dataset:
    if not mentions_openssl_cc(cert):
        # Check report text
        report_matches = scan_text_for_functions(cert.state.report.txt_path, all_functions)
        # Check target text
        st_matches = scan_text_for_functions(cert.state.st.txt_path, all_functions)
        
        # Combine matches
        all_matches = {**report_matches}
        for k, v in st_matches.items():
            all_matches[k] = all_matches.get(k, 0) + v
            
        if all_matches:
            results.append({
                'Dataset': 'CC',
                'ID': cert.dgst,
                'Manufacturer': cert.manufacturer,
                'Name': cert.name,
                'IssueDate': cert.not_valid_before,
                'MatchedFunctions': list(all_matches.keys()),
                'FunctionCount': len(all_matches),
                'TotalOccurrences': sum(all_matches.values())
            })
print(f"Found {len(results)} CC certificates mentioning OpenSSL functions.")

Scanning CC Dataset...
Found 5365 CC certificates mentioning OpenSSL functions.


### 4. Scan FIPS 140 Dataset

In [6]:
print("Scanning FIPS Dataset...")
for cert in fips_dataset:
    if not mentions_openssl_fips(cert):
        # FIPS uses policy_txt_path
        policy_matches = scan_text_for_functions(cert.state.policy_txt_path, all_functions)
        
        if policy_matches:
            results.append({
                'Dataset': 'FIPS',
                'ID': cert.dgst,
                'Manufacturer': cert.web_data.vendor if hasattr(cert, 'web_data') and cert.web_data else None,
                'Name': cert.web_data.module_name if hasattr(cert, 'web_data') and cert.web_data else None,
                'IssueDate': cert.web_data.validation_history[0].date if hasattr(cert, 'web_data') and cert.web_data and cert.web_data.validation_history else None,
                'MatchedFunctions': list(policy_matches.keys()),
                'FunctionCount': len(policy_matches),
                'TotalOccurrences': sum(policy_matches.values())
            })
print(f"Found {len(results)} FIPS certificates mentioning OpenSSL functions.")

Scanning FIPS Dataset...


AttributeError: 'InternalState' object has no attribute 'policy_txt_path'

### 5. Display Results

In [1]:
df_undisclosed = pd.DataFrame(results)

if not df_undisclosed.empty:
    df_undisclosed = df_undisclosed.sort_values(by='FunctionCount', ascending=False)
    for man in df_undisclosed['Manufacturer'].unique():
        man_df = df_undisclosed[df_undisclosed['Manufacturer'] == man]
        print(f"\nManufacturer: {man}")
    display(df_undisclosed.head(20))
    print(f"Found {len(df_undisclosed)} certificates using OpenSSL functions without mentioning OpenSSL.")
else:
    print("No undisclosed rebranding found.")


NameError: name 'pd' is not defined